# SASRec Stage 3 Baseline Multi-Task Weight-0.1 BPI2012 Colab Train 05

This notebook tests whether lowering `time_loss_weight` from `1.0` to `0.1`
helps the plain `anchor_ml20` multi-task model.

Main comparison groups:
- `anchor_single_task`
- `anchor_multi_task_w1.0`
- `anchor_multi_task_w0.1`

Main comparison metric:
- `full ranking + NDCG@10`


In [1]:
import torch

print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu name:', torch.cuda.get_device_name(0))


torch version: 2.11.0+cpu
cuda available: False


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
GITHUB_USERNAME = 'hwbuzz'

DRIVE_ROOT = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction'
REPO_DIR = '/content/time-aware-behavior-prediction'

DATA_DIR = f'{DRIVE_ROOT}/data/processed/bpi2012_complete_only_stage3_v2'
BASELINE_NDCG10_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10'
MULTITASK_BASELINE_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_baseline_multitask_ndcg10_v2'
MULTITASK_BASELINE_W01_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_baseline_multitask_w01_ndcg10_v2'
NOTEBOOK_DIR = f'{DRIVE_ROOT}/notebooks'

print('DATA_DIR:', DATA_DIR)
print('BASELINE_NDCG10_OUTPUT_DIR:', BASELINE_NDCG10_OUTPUT_DIR)
print('MULTITASK_BASELINE_OUTPUT_DIR:', MULTITASK_BASELINE_OUTPUT_DIR)
print('MULTITASK_BASELINE_W01_OUTPUT_DIR:', MULTITASK_BASELINE_W01_OUTPUT_DIR)
print('NOTEBOOK_DIR:', NOTEBOOK_DIR)


DATA_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/data/processed/bpi2012_complete_only_stage3_v2
BASELINE_NDCG10_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10
MULTITASK_BASELINE_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_baseline_multitask_ndcg10_v2
MULTITASK_BASELINE_W01_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_baseline_multitask_w01_ndcg10_v2
NOTEBOOK_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/notebooks


In [ ]:
!mkdir -p "$NOTEBOOK_DIR"
!mkdir -p "$DATA_DIR"
!mkdir -p "$BASELINE_NDCG10_OUTPUT_DIR"
!mkdir -p "$MULTITASK_BASELINE_OUTPUT_DIR"
!mkdir -p "$MULTITASK_BASELINE_W01_OUTPUT_DIR"


In [ ]:
%cd /content
!test -d time-aware-behavior-prediction || git clone https://github.com/$GITHUB_USERNAME/time-aware-behavior-prediction.git
%cd /content/time-aware-behavior-prediction
!git pull


In [ ]:
%cd /content/time-aware-behavior-prediction

skip_packages = ['pywinpty']

with open('requirements.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()

with open('requirements_colab.txt', 'w', encoding='utf-8') as f:
    for line in lines:
        pkg = line.strip().lower()
        if not any(name in pkg for name in skip_packages):
            f.write(line)

print('created requirements_colab.txt')


In [ ]:
!pip install -r requirements_colab.txt


## Prepare Stage 3 processed dataset

This notebook regenerates the Stage 3 dataset into the versioned Drive folder
before training, so the run does not depend on any stale local processed files.


In [ ]:
%cd /content/time-aware-behavior-prediction
!python scripts/regenerate_stage3_processed_dataset.py --output-dir "$DATA_DIR" --backup-existing
!ls "$DATA_DIR"


In [ ]:
%cd /content/time-aware-behavior-prediction
!mkdir -p data/processed
!rm -rf data/processed/bpi2012_complete_only_stage3_v2
!cp -r "$DATA_DIR" data/processed/
!ls data/processed/bpi2012_complete_only_stage3_v2


## Verify Stage 3 processed file

Stage 3 next-time prediction uses the processed time-feature CSV.
This check confirms that `delta_next_seconds` already exists.


In [ ]:
import pandas as pd

time_features_path = 'data/processed/bpi2012_complete_only_stage3_v2/events_encoded_time_features.csv'
df = pd.read_csv(time_features_path)
required_cols = [
    'delta_prev_seconds',
    'delta_start_seconds',
    'delta_next_seconds',
]
missing = [c for c in required_cols if c not in df.columns]

if missing:
    raise ValueError(f'Missing required Stage 3 columns: {missing}')

print('Stage 3 processed file is ready.')
print(df.columns.tolist())
df[['user_id', 'event_idx', 'delta_prev_seconds', 'delta_start_seconds', 'delta_next_seconds']].head()


## Experiment design

This experiment keeps the plain `anchor_ml20` multi-task model and only changes:
- `time_loss_weight: 1.0 -> 0.1`

Fixed settings:
- backbone: `anchor_ml20`
- multi-task outputs: `next activity + next time`
- time target: `delta_next_seconds`
- time target transform: `log1p`
- time loss: `huber`
- best epoch criterion: `full_valid_ndcg@10`
- final comparison uses all 3 seeds: `42`, `2024`, `7`


## Check prerequisite runs


In [ ]:
from pathlib import Path

baseline_required_runs = [
    'anchor_ml20_s42',
    'anchor_ml20_s2024',
    'anchor_ml20_s7',
]
multitask_required_runs = [
    'multitask_anchor_ml20_s42',
    'multitask_anchor_ml20_s2024',
    'multitask_anchor_ml20_s7',
]

print('=' * 80)
print('Single-task baseline prerequisite runs')
baseline_output_dir = Path(BASELINE_NDCG10_OUTPUT_DIR)
for run_name in baseline_required_runs:
    run_dir = baseline_output_dir / run_name
    print(run_name, 'EXISTS' if run_dir.exists() else 'MISSING')

print('=' * 80)
print('Existing plain multi-task w1.0 runs')
multitask_output_dir = Path(MULTITASK_BASELINE_OUTPUT_DIR)
for run_name in multitask_required_runs:
    run_dir = multitask_output_dir / run_name
    print(run_name, 'EXISTS' if run_dir.exists() else 'MISSING')


## Check planned w0.1 runs


In [ ]:
planned_multitask_w01_runs = [
    'multitask_anchor_ml20_w01_s42',
    'multitask_anchor_ml20_w01_s2024',
    'multitask_anchor_ml20_w01_s7',
]

output_dir = Path(MULTITASK_BASELINE_W01_OUTPUT_DIR)
print('=' * 80)
print('Stage 3 plain multi-task w0.1 runs')
for run_name in planned_multitask_w01_runs:
    run_dir = output_dir / run_name
    print(run_name, 'EXISTS' if run_dir.exists() else 'OK')


## Train plain multi-task w0.1 runs


In [ ]:
!python src/train_sasrec.py \
  --run_name multitask_anchor_ml20_w01_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 42 \
  --enable_time_prediction \
  --time_prediction_target delta_next_seconds \
  --time_target_transform log1p \
  --time_loss_type huber \
  --time_loss_weight 0.1 \
  --output_dir "$MULTITASK_BASELINE_W01_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only_stage3_v2/sasrec_interactions.txt \
  --time_features_path data/processed/bpi2012_complete_only_stage3_v2/events_encoded_time_features.csv \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


In [ ]:
!python src/train_sasrec.py \
  --run_name multitask_anchor_ml20_w01_s2024 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 2024 \
  --enable_time_prediction \
  --time_prediction_target delta_next_seconds \
  --time_target_transform log1p \
  --time_loss_type huber \
  --time_loss_weight 0.1 \
  --output_dir "$MULTITASK_BASELINE_W01_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only_stage3_v2/sasrec_interactions.txt \
  --time_features_path data/processed/bpi2012_complete_only_stage3_v2/events_encoded_time_features.csv \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


In [ ]:
!python src/train_sasrec.py \
  --run_name multitask_anchor_ml20_w01_s7 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 7 \
  --enable_time_prediction \
  --time_prediction_target delta_next_seconds \
  --time_target_transform log1p \
  --time_loss_type huber \
  --time_loss_weight 0.1 \
  --output_dir "$MULTITASK_BASELINE_W01_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only_stage3_v2/sasrec_interactions.txt \
  --time_features_path data/processed/bpi2012_complete_only_stage3_v2/events_encoded_time_features.csv \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


In [4]:
from pathlib import Path
import json
import pandas as pd

def rebuild_df(output_dir: str):
    rows = []
    output_path = Path(output_dir)
    if not output_path.exists():
        return pd.DataFrame()
    for run_dir in output_path.iterdir():
        if not run_dir.is_dir():
            continue
        summary_path = run_dir / 'metrics_summary.json'
        config_path = run_dir / 'config.json'
        if not summary_path.exists() or not config_path.exists():
            continue
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
        config = json.loads(config_path.read_text(encoding='utf-8'))
        row = {
            'run_name': summary.get('run_name'),
            'run_dir': str(run_dir),
            'completed_at': summary.get('completed_at'),
            'best_epoch': summary.get('best_epoch'),
            'checkpoint_best': summary.get('checkpoint_best'),
            'checkpoint_last': summary.get('checkpoint_last'),
            'metrics_history': summary.get('metrics_history'),
            'config_path': str(config_path),
            'metrics_summary': str(summary_path),
            'maxlen': config.get('maxlen'),
            'dropout_rate': config.get('dropout_rate'),
            'hidden_units': config.get('hidden_units'),
            'seed': config.get('seed'),
            'selection_metric': config.get('selection_metric'),
            'enable_time_prediction': config.get('enable_time_prediction', False),
            'time_prediction_target': config.get('time_prediction_target'),
            'time_loss_weight': config.get('time_loss_weight'),
            'time_target_transform': config.get('time_target_transform'),
            'time_modeling_mode': config.get('time_modeling_mode'),
        }
        for group_name in ['best_valid', 'best_test_at_best_valid', 'last_valid', 'last_test']:
            group = summary.get(group_name) or {}
            for mode, metrics in group.items():
                for key, value in metrics.items():
                    row[f'{group_name}_{mode}_{key}'] = value
        rows.append(row)
    return pd.DataFrame(rows)


In [5]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1600)
pd.set_option('display.max_colwidth', None)


## Comparison summary


In [6]:
baseline_runs = [
    'anchor_ml20_s42',
    'anchor_ml20_s2024',
    'anchor_ml20_s7',
]
multitask_w10_runs = [
    'multitask_anchor_ml20_s42',
    'multitask_anchor_ml20_s2024',
    'multitask_anchor_ml20_s7',
]
multitask_w01_runs = [
    'multitask_anchor_ml20_w01_s42',
    'multitask_anchor_ml20_w01_s2024',
    'multitask_anchor_ml20_w01_s7',
]

baseline_df = rebuild_df(BASELINE_NDCG10_OUTPUT_DIR)
multitask_w10_df = rebuild_df(MULTITASK_BASELINE_OUTPUT_DIR)
multitask_w01_df = rebuild_df(MULTITASK_BASELINE_W01_OUTPUT_DIR)

baseline_subset = baseline_df[baseline_df['run_name'].isin(baseline_runs)].copy()
baseline_subset['variant'] = 'anchor_single_task'

multitask_w10_subset = multitask_w10_df[multitask_w10_df['run_name'].isin(multitask_w10_runs)].copy()
multitask_w10_subset['variant'] = 'anchor_multi_task_w1.0'

multitask_w01_subset = multitask_w01_df[multitask_w01_df['run_name'].isin(multitask_w01_runs)].copy()
multitask_w01_subset['variant'] = 'anchor_multi_task_w0.1'

df_compare = pd.concat(
    [baseline_subset, multitask_w10_subset, multitask_w01_subset],
    ignore_index=True,
)
df_compare = df_compare.sort_values(['variant', 'seed', 'run_name']).reset_index(drop=True)

id_cols = [
    'run_name', 'seed', 'variant', 'maxlen', 'dropout_rate',
    'selection_metric', 'time_loss_weight',
]

metric_prefixes = (
    'best_valid_',
    'best_test_at_best_valid_',
    'last_valid_',
    'last_test_',
)

metric_cols = sorted([
    c for c in df_compare.columns
    if c.startswith(metric_prefixes)
])

display_cols = [c for c in id_cols if c in df_compare.columns] + metric_cols
df_compare[display_cols]


/tmp/ipykernel_723/1576682939.py:30: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_compare = pd.concat(


,run_name,seed,variant,maxlen,dropout_rate,selection_metric,time_loss_weight,best_test_at_best_valid_full_hr@10,best_test_at_best_valid_full_hr@5,best_test_at_best_valid_full_mean_rank,best_test_at_best_valid_full_median_rank,best_test_at_best_valid_full_mrr,best_test_at_best_valid_full_ndcg@10,best_test_at_best_valid_full_ndcg@5,best_test_at_best_valid_full_num_eval_users,best_test_at_best_valid_sampled_hr@10,best_test_at_best_valid_sampled_hr@5,best_test_at_best_valid_sampled_mean_rank,best_test_at_best_valid_sampled_median_rank,best_test_at_best_valid_sampled_mrr,best_test_at_best_valid_sampled_ndcg@10,best_test_at_best_valid_sampled_ndcg@5,best_test_at_best_valid_sampled_num_eval_users,best_test_at_best_valid_task_accuracy,best_test_at_best_valid_task_macro_f1,best_test_at_best_valid_task_time_mae,best_test_at_best_valid_task_time_median_ae,best_test_at_best_valid_task_time_rmse,best_test_at_best_valid_task_top10_accuracy,best_test_at_best_valid_task_top1_accuracy,best_test_at_best_valid_task_top5_accuracy,best_valid_full_hr@10,best_valid_full_hr@5,best_valid_full_mean_rank,best_valid_full_median_rank,best_valid_full_mrr,best_valid_full_ndcg@10,best_valid_full_ndcg@5,best_valid_full_num_eval_users,best_valid_sampled_hr@10,best_valid_sampled_hr@5,best_valid_sampled_mean_rank,best_valid_sampled_median_rank,best_valid_sampled_mrr,best_valid_sampled_ndcg@10,best_valid_sampled_ndcg@5,best_valid_sampled_num_eval_users,best_valid_task_accuracy,best_valid_task_macro_f1,best_valid_task_time_mae,best_valid_task_time_median_ae,best_valid_task_time_rmse,best_valid_task_top10_accuracy,best_valid_task_top1_accuracy,best_valid_task_top5_accuracy,last_test_full_hr@10,last_test_full_hr@5,last_test_full_mean_rank,last_test_full_median_rank,last_test_full_mrr,last_test_full_ndcg@10,last_test_full_ndcg@5,last_test_full_num_eval_users,last_test_sampled_hr@10,last_test_sampled_hr@5,last_test_sampled_mean_rank,last_test_sampled_median_rank,last_test_sampled_mrr,last_test_sampled_ndcg@10,last_test_sampled_ndcg@5,last_test_sampled_num_eval_users,last_test_task_accuracy,last_test_task_macro_f1,last_test_task_time_mae,last_test_task_time_median_ae,last_test_task_time_rmse,last_test_task_top10_accuracy,last_test_task_top1_accuracy,last_test_task_top5_accuracy,last_valid_full_hr@10,last_valid_full_hr@5,last_valid_full_mean_rank,last_valid_full_median_rank,last_valid_full_mrr,last_valid_full_ndcg@10,last_valid_full_ndcg@5,last_valid_full_num_eval_users,last_valid_sampled_hr@10,last_valid_sampled_hr@5,last_valid_sampled_mean_rank,last_valid_sampled_median_rank,last_valid_sampled_mrr,last_valid_sampled_ndcg@10,last_valid_sampled_ndcg@5,last_valid_sampled_num_eval_users,last_valid_task_accuracy,last_valid_task_macro_f1,last_valid_task_time_mae,last_valid_task_time_median_ae,last_valid_task_time_rmse,last_valid_task_top10_accuracy,last_valid_task_top1_accuracy,last_valid_task_top5_accuracy
0,multitask_anchor_ml20_w01_s7,7,anchor_multi_task_w0.1,20,0.2,full_valid_ndcg@10,0.1,1.000000,0.957722,1.901577,2.0,0.665485,0.750997,0.736530,7356,0.298260,0.156199,15.219549,14.0,0.205477,0.192348,0.148220,7356,0.030587,0.025439,14238.078419,73.410313,75779.808711,0.693719,0.030587,0.468189,0.963757,0.857880,2.842270,1.0,0.685248,0.749368,0.716602,7367,0.613818,0.563051,16.070042,1.0,0.581278,0.575630,0.559582,7367,0.084838,0.135210,84201.458773,6258.683644,286108.613489,0.793674,0.084838,0.405593,1.000000,0.997284,1.973652,1.0,0.723784,0.791712,0.790772,7363,0.402282,0.340622,14.162298,15.0,0.368154,0.351943,0.332377,7363,0.107972,0.041378,14121.966541,121.081741,67066.787774,0.708950,0.107972,0.268912,0.939908,0.824742,3.142431,1.0,0.676202,0.734823,0.699485,7372,0.616386,0.563755,17.587086,1.0,0.578130,0.575277,0.558619,7372,0.054395,0.064740,77514.318551,7966.484646,282558.143050,0.756240,0.054395,0.401926
1,multitask_anchor_ml20_w01_s42,42,anchor_multi_task_w0.1,20,0.2,full_valid_ndcg@10,0.1,1.000000,0.948477,1.583197,1.0,0.871299,0.902311,0.885558,7356,0.580886,

In [7]:
summary_metric_cols = sorted([
    c for c in df_compare.columns
    if c.startswith((
        'best_valid_',
        'best_test_at_best_valid_',
        'last_valid_',
        'last_test_',
    ))
])

summary_compare = df_compare.groupby('variant')[summary_metric_cols].agg(['mean', 'std'])
summary_compare


best_test_at_best_valid_full_hr@10           best_test_at_best_valid_full_hr@5           best_test_at_best_valid_full_mean_rank           best_test_at_best_valid_full_median_rank          best_test_at_best_valid_full_mrr           best_test_at_best_valid_full_ndcg@10           best_test_at_best_valid_full_ndcg@5           best_test_at_best_valid_full_num_eval_users            best_test_at_best_valid_sampled_hr@10           best_test_at_best_valid_sampled_hr@5           best_test_at_best_valid_sampled_mean_rank           best_test_at_best_valid_sampled_median_rank           best_test_at_best_valid_sampled_mrr           best_test_at_best_valid_sampled_ndcg@10           best_test_at_best_valid_sampled_ndcg@5           best_test_at_best_valid_sampled_num_eval_users            best_test_at_best_valid_task_accuracy           best_test_at_best_valid_task_macro_f1           best_test_at_best_valid_task_time_mae              best_test_at_best_valid_task_time_median_ae             best_test_at_best_valid_task_time_rmse              best_test_at_best_valid_task_top10_accuracy           best_test_at_best_valid_task_top1_accuracy           best_test_at_best_valid_task_top5_accuracy           best_valid_full_hr@10           best_valid_full_hr@5           best_valid_full_mean_rank           best_valid_full_median_rank      best_valid_full_mrr           best_valid_full_ndcg@10           best_valid_full_ndcg@5           best_valid_full_num_eval_users            best_valid_sampled_hr@10           best_valid_sampled_hr@5           best_valid_sampled_mean_rank  \
                                                     mean       std                              mean       std                                   mean       std                                     mean      std                             mean       std                                 mean       std                                mean       std                                        mean        std                                  mean       std                                 mean       std                                      mean       std                                        mean       std                                mean       std                                    mean       std                                   mean       std                                           mean        std                                  mean       std                                  mean       std                                  mean          std                                        mean         std                                   mean          std                                        mean       std                                       mean       std                                       mean       std                  mean       std                 mean       std                      mean       std                        mean  std                mean       std                    mean       std                   mean       std                           mean        std                     mean       std                    mean       std                         mean   
variant                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

Interpretation guide:

- compare `anchor_multi_task_w1.0` vs `anchor_multi_task_w0.1` first
- use `best_test_at_best_valid_full_ndcg@10` as the main decision metric
- if `w0.1` improves ranking without overly damaging time error, then the plain multitask drop was partly a loss-balance issue
